In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
'''
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
'''
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

"\nfor dirname, _, filenames in os.walk('/kaggle/input'):\n    for filename in filenames:\n        print(os.path.join(dirname, filename))\n"

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

2026-02-16 10:41:37.550092: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771238497.787692      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771238497.854183      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771238498.403700      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771238498.403750      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771238498.403753      24 computation_placer.cc:177] computation placer alr

In [3]:
dataset_dir = "/kaggle/input/microsoft-catsvsdogs-dataset/PetImages"

In [4]:
image_size = (180, 180)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=image_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=image_size,
    batch_size=batch_size
)


Found 25000 files belonging to 2 classes.
Using 20000 files for training.


I0000 00:00:1771238547.869219      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1771238547.875477      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 25000 files belonging to 2 classes.
Using 5000 files for validation.


In [5]:
class_names = train_ds.class_names
class_names


['Cat', 'Dog']

In [6]:
normalization_layer = layers.Rescaling(1./255)


In [7]:
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds   = val_ds.map(lambda x, y: (normalization_layer(x), y))

train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
val_ds   = val_ds.apply(tf.data.experimental.ignore_errors())


Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


In [8]:
model = keras.Sequential()

model.add(layers.Input(shape=(180, 180, 3)))

model.add(layers.Conv2D(16, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Conv2D(32, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Flatten())
model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dropout(0.5))   # <<< EKLENDİ
model.add(layers.Dense(1, activation='sigmoid'))

In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 178, 178, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 89, 89, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 87, 87, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 43, 43, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 59168)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │     1,893,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,898,529 (7.24 MB)

 Trainable params: 1,898,529 (7.24 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])


In [11]:
history = model.fit(train_ds,validation_data=val_ds,epochs=5)


Epoch 1/5


I0000 00:00:1771238560.575827      74 service.cc:152] XLA service 0x788ee800a700 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1771238560.575861      74 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1771238560.575865      74 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
Corrupt JPEG data: 162 extraneous bytes before marker 0xd9
I0000 00:00:1771238561.043526      74 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-02-16 10:42:42.724518: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-02-16 10:42:42.863444: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsig

     12/Unknown 6s 16ms/step - accuracy: 0.4244 - loss: 1.4781

I0000 00:00:1771238564.676248      74 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


    181/Unknown 14s 49ms/step - accuracy: 0.4943 - loss: 0.8270

...

593/618 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.5005 - loss: 0.6932

Corrupt JPEG data: 1403 extraneous bytes before marker 0xd9


616/618 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.5003 - loss: 0.6932

Corrupt JPEG data: 65 extraneous bytes before marker 0xd9
Corrupt JPEG data: 239 extraneous bytes before marker 0xd9
Corrupt JPEG data: 214 extraneous bytes before marker 0xd9


618/618 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.5003 - loss: 0.6932 - val_accuracy: 0.4919 - val_loss: 0.6932


In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 178, 178, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 89, 89, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 87, 87, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 43, 43, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 59168)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │     1,893,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,695,589 (21.73 MB)

 Trainable params: 1,898,529 (7.24 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 3,797,060 (14.48 MB)

In [13]:
from sklearn.metrics import f1_score, classification_report
import numpy as np

# 1. Tahminleri alıyoruz (Senin değişkenin val_ds olduğu için)
Y_pred = model.predict(val_ds)
y_pred = (Y_pred > 0.5).astype(int).flatten()

# 2. val_ds içinden gerçek etiketleri (y_true) çekiyoruz
# TensorFlow Dataset'ten etiketleri almanın en temiz yolu budur:
y_true = np.concatenate([y for x, y in val_ds], axis=0)

# 3. Sonuçları yazdıralım
f1 = f1_score(y_true, y_pred)
print(f"--- KESİN SONUÇLAR ---")
print(f"Genel F1 Score: {f1:.4f}")
print("\nSınıf Bazlı Detaylı Rapor:")
print(classification_report(y_true, y_pred, target_names=['Cat', 'Dog']))

     53/Unknown 1s 19ms/step

Corrupt JPEG data: 65 extraneous bytes before marker 0xd9


     74/Unknown 2s 20ms/step

Corrupt JPEG data: 239 extraneous bytes before marker 0xd9


    137/Unknown 3s 21ms/step

Corrupt JPEG data: 214 extraneous bytes before marker 0xd9


155/155 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step


Corrupt JPEG data: 65 extraneous bytes before marker 0xd9
Corrupt JPEG data: 239 extraneous bytes before marker 0xd9
Corrupt JPEG data: 214 extraneous bytes before marker 0xd9


--- KESİN SONUÇLAR ---
Genel F1 Score: 0.0000

Sınıf Bazlı Detaylı Rapor:
              precision    recall  f1-score   support

         Cat       0.49      1.00      0.66      2432
         Dog       0.00      0.00      0.00      2504

    accuracy                           0.49      4936
   macro avg       0.25      0.50      0.33      4936
weighted avg       0.24      0.49      0.33      4936



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
